# Notebook 04 · Image preparation, part 3: common size and MONAI/YOLO export

Prerequisite: notebooks 02 and 03 already run. Runs on a CPU (no GPU); it opens every image, so it takes a
while (20 to 40 minutes).

What this part does, in order:

1. **Common size for all images:** each slice is resampled to a pixel of **0.488 mm** (CQ500 median) and
   cropped to **512 × 512** (25 cm) **centred on the head** of the examination. Areas outside the original
   image are filled with air (−1,024 HU). The boxes go through the same transformation.
2. **Three windows as three channels** (brain 40/80, subdural 80/200, bone 600/2800), stored as 8-bit PNG
   files: **a single file per slice, read by both detectors**.
3. **Labels in both formats**, from the same box table: MONAI JSON (box in pixels, x1 y1 x2 y2) and YOLO
   `.txt` (centre, width and height from 0 to 1). Training uses **one class** (haemorrhage); the subtype of
   each box is kept in the table.
4. **Lists of the five folds** (training and validation on BHX) and of the test set (CT-ICH), in both
   formats.
5. **Automated tests:** the YOLO boxes, converted back, must equal the MONAI boxes; each PNG read back must
   equal the array written; no box may be clipped at the image border.
6. **Figure** showing a slice before and after preparation.

Outputs on the Drive (`tc/preparo/`): `cq500_512.zip` and `ctich_512.zip` (images, labels and lists) and the
summaries. The CT-ICH images and labels must not be shared (data use agreement).

**Language:** documentation and comments are in English; identifiers, printed messages and the column
names of the files written here are in Portuguese, exactly as in the run that produced the results.

In [ ]:
# 1. Folders, libraries and parameters
import csv, hashlib, io, json, os, shutil, subprocess, sys, time, zipfile, collections
import concurrent.futures as cf
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DADOS = Path('/content/drive/MyDrive/ich-benchmark/dados')
    LOCAL = Path('/content/preparo_512')             # Colab disk: many small files are slow on the Drive
    subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'pydicom', 'nibabel', 'pylibjpeg',
                    'pylibjpeg-libjpeg'], check=True)
except ImportError:
    DADOS = Path(os.environ['PREPARO_LOCAL'])
    LOCAL = DADOS / 'preparo_512_local'

import numpy as np
import pydicom   # with pylibjpeg: some CQ500 examinations are stored as lossless JPEG
import nibabel as nib
from scipy import ndimage
from PIL import Image

CQ500, CTICH, PREPARO = DADOS / 'tc' / 'cq500', DADOS / 'tc' / 'ct-ich', DADOS / 'tc' / 'preparo'
PIXEL_MM = 0.488281                     # common pixel size (CQ500 median); 512 px = 250 mm
LADO = 512
AR_HU = -1024.0                         # value outside the original image
JANELAS = {'brain': (40, 80), 'subdural': (80, 200), 'bone': (600, 2800)}   # level, width in HU
CABECA_HU = -300                        # above this is tissue (to find the head)
CLASSE = 0                              # one class: "hemorrhage"
shutil.rmtree(LOCAL, ignore_errors=True)
for base in ('cq500', 'ctich'):
    for sub in ('images', 'labels', 'monai', 'yolo'):
        (LOCAL / base / sub).mkdir(parents=True, exist_ok=True)
print(DADOS, LOCAL, sep='\n')

In [ ]:
# 2. Preparation functions: resample and crop around the head, windows, boxes
def janelas(hu):
    """HU (H × W) → 3 canais de 0 a 1 (cérebro, subdural, osso)."""
    return np.stack([np.clip((hu - (n - l / 2)) / l, 0, 1) for n, l in JANELAS.values()], axis=-1)

def centro_da_cabeca(volume):
    """Centro (linha, coluna) e tamanho (altura, largura) da cabeça, em pixels da imagem original.
    Em cada corte (numa cópia reduzida 4×, por velocidade): tecido acima de −300 HU, sem estruturas finas
    (apoio de cabeça, mesa), maior região. Centro = mediana dos centros dos cortes; tamanho = o maior."""
    centros, tamanhos = [], []
    for corte in volume[:, ::4, ::4]:
        tecido = ndimage.binary_opening(corte > CABECA_HU, structure=np.ones((4, 4)))
        rotulos, n = ndimage.label(tecido)
        if n == 0:
            continue
        maior = 1 + int(np.argmax(ndimage.sum(tecido, rotulos, range(1, n + 1))))
        linhas, colunas = np.nonzero(rotulos == maior)
        centros.append(((linhas.min() + linhas.max() + 1) * 2, (colunas.min() + colunas.max() + 1) * 2))
        tamanhos.append(((linhas.max() + 1 - linhas.min()) * 4, (colunas.max() + 1 - colunas.min()) * 4))
    if not centros:
        return (volume.shape[1] / 2, volume.shape[2] / 2), (0, 0)
    c = np.median(np.array(centros, dtype=float), axis=0)
    t = np.array(tamanhos).max(axis=0)
    return (float(c[0]), float(c[1])), (int(t[0]), int(t[1]))

def geometria(pixel_mm, centro):
    """Escala e deslocamento: coordenada contínua de saída = entrada × escala − deslocamento."""
    escala = pixel_mm / PIXEL_MM
    return escala, (centro[0] * escala - LADO / 2, centro[1] * escala - LADO / 2)

def reamostrar(hu, escala, desloc):
    # output pixel q (centre at q + 0.5) comes from the continuous input position (q + 0.5 + offset) / scale
    off = [(0.5 + d) / escala - 0.5 for d in desloc]
    return ndimage.affine_transform(hu, [1 / escala, 1 / escala], offset=off, output_shape=(LADO, LADO),
                                    order=1, mode='constant', cval=AR_HU)

cortadas = []                                       # boxes beyond the border (should stay empty)

def caixa_512(x, y, w, h, escala, desloc, quem):
    """Caixa (x, y, largura, altura) da imagem original → (x1, y1, x2, y2) na imagem de 512. Se passar da
    borda, é cortada no limite e registrada em `cortadas` com a fração da área que se perdeu."""
    c = (x * escala - desloc[1], y * escala - desloc[0], (x + w) * escala - desloc[1], (y + h) * escala - desloc[0])
    d = tuple(min(max(v, 0.0), float(LADO)) for v in c)
    if d != c:
        area = lambda b: max(b[2] - b[0], 0) * max(b[3] - b[1], 0)
        cortadas.append({'onde': quem, 'fracao_perdida': round(1 - area(d) / area(c), 3)})
    return d

def gravar(base, nome_arquivo, hu, escala, desloc):
    img = np.round(janelas(reamostrar(hu, escala, desloc)) * 255).astype(np.uint8)
    caminho = LOCAL / base / 'images' / nome_arquivo
    Image.fromarray(img).save(caminho, compress_level=1)
    return hashlib.sha1(img.tobytes()).hexdigest()

In [ ]:
# 3. CQ500: read each examination (selected series), find the head, write the slices and convert the boxes
escolhidas = {l['Name']: l for l in csv.DictReader(open(PREPARO / 'cq500_series_escolhidas.csv'))}
particao = {l['Name']: l for l in csv.DictReader(open(PREPARO / 'particao_5partes.csv'))}
caixas_bhx = collections.defaultdict(list)
for l in csv.DictReader(open(PREPARO / 'cq500_caixas.csv')):
    caixas_bhx[(l['Name'], int(l['corte']))].append(l)

def zip_do_exame(nome):
    recuperado = CQ500 / 'zips_recuperados' / f'{nome}.zip'
    return recuperado if recuperado.exists() else CQ500 / 'zips' / f'{nome}.zip'

def preparar_cq500(nome):
    e = escolhidas[nome]
    with zipfile.ZipFile(zip_do_exame(nome)) as z:
        dcms = [pydicom.dcmread(io.BytesIO(z.read(m))) for m in z.namelist()
                if m.lower().endswith('.dcm') and m.rsplit('/', 2)[-2] == e['pasta']]
    dcms.sort(key=lambda d: float(d.ImagePositionPatient[2]))
    vol = np.stack([d.pixel_array * float(d.RescaleSlope) + float(d.RescaleIntercept) for d in dcms]).astype(np.float32)
    pixel = float(dcms[0].PixelSpacing[0])
    centro, tamanho = centro_da_cabeca(vol)
    escala, desloc = geometria(pixel, centro)
    imagens, caixas = [], []
    for k in range(len(vol)):
        arq = f'{nome}_s{k:03d}.png'
        sha = gravar('cq500', arq, vol[k], escala, desloc)
        imagens.append({'base': 'cq500', 'exame': nome, 'corte': k, 'arquivo': arq, 'grupo': e['grupo'],
                        'parte': particao[nome]['parte'], 'sha1': sha})
        for c in caixas_bhx.get((nome, k), []):
            x1, y1, x2, y2 = caixa_512(float(c['x']), float(c['y']), float(c['largura']), float(c['altura']),
                                       escala, desloc, f'{nome} s{k}')
            if x2 <= x1 or y2 <= y1:
                continue
            caixas.append({'base': 'cq500', 'exame': nome, 'corte': k, 'arquivo': arq, 'tipo': c['tipo'],
                           'x1': x1, 'y1': y1, 'x2': x2, 'y2': y2})
    info = {'exame': nome, 'pixel_mm_original': pixel, 'cortes': len(vol), 'escala': escala,
            'cabeca_mm': [round(float(t) * pixel, 1) for t in tamanho]}
    return imagens, caixas, info

nomes = sorted(particao, key=lambda n: int(n.rsplit('-', 1)[1]))
nomes = [n for n in nomes if zip_do_exame(n).exists()]                   # in Colab, all of them
inicio = time.time()
img_cq, cx_cq, info_cq = [], [], []
with cf.ThreadPoolExecutor(4) as ex:
    for i, (a, b, c) in enumerate(ex.map(preparar_cq500, nomes), start=1):
        img_cq += a; cx_cq += b; info_cq.append(c)
        if i % 50 == 0:
            print(f'{i}/{len(nomes)} exames, {time.time() - inicio:.0f} s')
print(len(nomes), 'exames;', len(img_cq), 'cortes;', len(cx_cq), f'caixas; {time.time() - inicio:.0f} s')

In [ ]:
# 4. CT-ICH: same transformations (NIfTI in the CQ500 orientation, as in notebook 03)
def para_dicom(volume):
    return np.transpose(volume[::-1, ::-1, :], (2, 1, 0))

def abrir(caminho):
    img = nib.as_closest_canonical(nib.load(caminho))
    return para_dicom(img.get_fdata(dtype=np.float32)), [float(x) for x in img.header.get_zooms()[:3]]

volumes = sorted(p for p in CTICH.rglob('*.nii*') if 'mask' not in str(p).lower())
caixas_ct = collections.defaultdict(list)
for l in csv.DictReader(open(CTICH / 'ctich_caixas.csv')):
    caixas_ct[(l['exame'], int(l['corte']))].append(l)

def preparar_ctich(v):
    exame = v.name.split('.')[0]
    vol, espac = abrir(v)
    centro, tamanho = centro_da_cabeca(vol)
    escala, desloc = geometria(espac[0], centro)
    imagens, caixas = [], []
    for k in range(len(vol)):
        arq = f'CTICH-{exame}_s{k:03d}.png'
        sha = gravar('ctich', arq, vol[k], escala, desloc)
        imagens.append({'base': 'ctich', 'exame': exame, 'corte': k, 'arquivo': arq, 'grupo': 'externo',
                        'parte': 'externo', 'sha1': sha})
        for c in caixas_ct.get((exame, k), []):
            x1, y1, x2, y2 = caixa_512(float(c['x']), float(c['y']), float(c['largura']), float(c['altura']),
                                       escala, desloc, f'CT-ICH {exame} s{k}')
            if x2 <= x1 or y2 <= y1:
                continue
            caixas.append({'base': 'ctich', 'exame': exame, 'corte': k, 'arquivo': arq, 'tipo': 'Hemorrhage',
                           'x1': x1, 'y1': y1, 'x2': x2, 'y2': y2})
    return imagens, caixas, {'exame': exame, 'pixel_mm_original': espac[0], 'cortes': len(vol), 'escala': escala,
                             'cabeca_mm': [round(float(t) * espac[0], 1) for t in tamanho]}

inicio = time.time()
img_ct, cx_ct, info_ct = [], [], []
with cf.ThreadPoolExecutor(4) as ex:
    for a, b, c in ex.map(preparar_ctich, volumes):
        img_ct += a; cx_ct += b; info_ct.append(c)
print(len(volumes), 'exames;', len(img_ct), 'cortes;', len(cx_ct), f'caixas; {time.time() - inicio:.0f} s')

In [ ]:
# 5. Labels in both formats and fold lists, from the SAME box table
def exportar(base, imagens, caixas):
    por_arquivo = collections.defaultdict(list)
    for c in caixas:
        por_arquivo[c['arquivo']].append(c)
    entradas = {}
    for im in imagens:
        cs = por_arquivo.get(im['arquivo'], [])
        # YOLO: one line per box "class xc yc w h", from 0 to 1; empty file = slice without haemorrhage
        linhas = [f'{CLASSE} {(c["x1"] + c["x2"]) / 2 / LADO:.6f} {(c["y1"] + c["y2"]) / 2 / LADO:.6f} '
                  f'{(c["x2"] - c["x1"]) / LADO:.6f} {(c["y2"] - c["y1"]) / LADO:.6f}' for c in cs]
        (LOCAL / base / 'labels' / im['arquivo'].replace('.png', '.txt')).write_text('\n'.join(linhas) + ('\n' if linhas else ''))
        # MONAI: box in pixels (x1, y1, x2, y2) and class label
        entradas[im['arquivo']] = {'image': f'images/{im["arquivo"]}',
                                   'box': [[round(c[k], 3) for k in ('x1', 'y1', 'x2', 'y2')] for c in cs],
                                   'label': [CLASSE] * len(cs), 'exam': im['exame'], 'slice': im['corte']}
    return entradas

ent_cq = exportar('cq500', img_cq, cx_cq)
ent_ct = exportar('ctich', img_ct, cx_ct)

def listas(base, nome, treino, validacao):
    (LOCAL / base / 'monai' / f'{nome}.json').write_text(json.dumps(
        {'training': [ent[a] for ent, a in treino], 'validation': [ent[a] for ent, a in validacao]}))
    for sufixo, grupo in (('train', treino), ('val', validacao)):
        (LOCAL / base / 'yolo' / f'{nome}_{sufixo}.txt').write_text(''.join(f'./images/{a}\n' for _, a in grupo))
    (LOCAL / base / 'yolo' / f'{nome}.yaml').write_text(
        f'path: .\ntrain: yolo/{nome}_train.txt\nval: yolo/{nome}_val.txt\nnames:\n  0: hemorrhage\n')

principal = [im for im in img_cq if im['grupo'] in ('com sangramento', 'normal')]   # equivocal examinations excluded
for k in range(1, 6):
    listas('cq500', f'fold{k}', [(ent_cq, im['arquivo']) for im in principal if im['parte'] != str(k)],
           [(ent_cq, im['arquivo']) for im in principal if im['parte'] == str(k)])
listas('cq500', 'equivocal', [], [(ent_cq, im['arquivo']) for im in img_cq if im['grupo'] == 'duvidoso'])
listas('ctich', 'external', [], [(ent_ct, im['arquivo']) for im in img_ct])

for base, imagens, caixas in (('cq500', img_cq, cx_cq), ('ctich', img_ct, cx_ct)):
    with open(LOCAL / base / 'index.csv', 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=imagens[0].keys()); w.writeheader(); w.writerows(imagens)
    with open(LOCAL / base / 'boxes_512.csv', 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=caixas[0].keys()); w.writeheader(); w.writerows(caixas)
print('Listas:', sorted(p.name for p in (LOCAL / 'cq500' / 'monai').iterdir()), sorted(p.name for p in (LOCAL / 'ctich' / 'monai').iterdir()))

In [ ]:
# 6. Automated tests (they stop the notebook if anything fails)
def testar(base, imagens, caixas, ent):
    r = {'images': len(imagens), 'boxes': len(caixas)}
    # (a) YOLO converted back = MONAI, box by box (tolerance 0.01 pixel)
    dif = 0.0
    for im in imagens:
        linhas = (LOCAL / base / 'labels' / im['arquivo'].replace('.png', '.txt')).read_text().split()
        yolo = np.array(linhas, dtype=float).reshape(-1, 5)
        monai = np.array(ent[im['arquivo']]['box'], dtype=float).reshape(-1, 4)
        assert len(yolo) == len(monai), im['arquivo']
        if len(yolo):
            xc, yc, w, h = (yolo[:, 1:] * LADO).T
            dif = max(dif, float(np.abs(np.stack([xc - w / 2, yc - h / 2, xc + w / 2, yc + h / 2], 1) - monai).max()))
    assert dif < 0.01, dif
    r['max_box_difference_yolo_vs_monai_px'] = round(dif, 5)
    # (b) each PNG read back is identical to what was written (sample of 200 slices)
    amostra = imagens[:: max(1, len(imagens) // 200)]
    for im in amostra:
        lido = np.asarray(Image.open(LOCAL / base / 'images' / im['arquivo']))
        assert lido.shape == (LADO, LADO, 3) and hashlib.sha1(lido.tobytes()).hexdigest() == im['sha1'], im['arquivo']
    r['png_reread_identical'] = f'{len(amostra)}/{len(amostra)}'
    # (c) all boxes inside the 512 image (those beyond the border were clipped in cell 3/4)
    assert all(0 <= c['x1'] < c['x2'] <= LADO and 0 <= c['y1'] < c['y2'] <= LADO for c in caixas)
    r['boxes_clipped_at_border'] = sum(1 for c in cortadas if c['onde'].startswith('CT-ICH') == (base == 'ctich'))
    # (d) same file list in both formats
    for k in ([f'fold{i}' for i in range(1, 6)] if base == 'cq500' else ['external']):
        m = json.loads((LOCAL / base / 'monai' / f'{k}.json').read_text())
        y = (LOCAL / base / 'yolo' / f'{k}_val.txt').read_text().split()
        assert [e['image'] for e in m['validation']] == [p[2:] for p in y], k
    r['same_file_lists_monai_yolo'] = True
    return r

testes = {'cq500': testar('cq500', img_cq, cx_cq, ent_cq), 'ctich': testar('ctich', img_ct, cx_ct, ent_ct)}
print(json.dumps(testes, indent=2))
if cortadas:
    print('Caixas cortadas na borda:', cortadas[:20])

In [ ]:
# 7. Summary (in English) and copy to the Drive
def resumo_base(imagens, caixas, info):
    esc = [i['escala'] for i in info]
    cab = [max(i['cabeca_mm']) for i in info]
    lados = [max(c['x2'] - c['x1'], c['y2'] - c['y1']) * PIXEL_MM for c in caixas]
    return {'exams': len(info), 'slices': len(imagens), 'slices_with_boxes': len({c['arquivo'] for c in caixas}),
            'boxes': len(caixas),
            'original_pixel_mm_range': [min(i['pixel_mm_original'] for i in info), max(i['pixel_mm_original'] for i in info)],
            'scale_factor_range': [round(min(esc), 3), round(max(esc), 3)],
            'head_extent_mm_median_max': [float(np.median(cab)), float(max(cab))],
            'heads_wider_than_field_of_view': int(sum(c > LADO * PIXEL_MM for c in cab)),
            'box_longest_side_mm_median_p5_p95': [round(float(np.percentile(lados, p)), 1) for p in (50, 5, 95)] if lados else None}

por_parte = {f'fold{k}': {'exams': len({i['exame'] for i in principal if i['parte'] == str(k)}),
                          'slices': sum(i['parte'] == str(k) for i in principal),
                          'slices_with_boxes': len({c['arquivo'] for c in cx_cq if particao[c['exame']]['parte'] == str(k)
                                                    and particao[c['exame']]['grupo'] == 'com sangramento'})}
             for k in range(1, 6)}
resumo = {
    'pixel_spacing_mm': PIXEL_MM, 'image_size_px': LADO, 'field_of_view_mm': round(LADO * PIXEL_MM, 1),
    'interpolation': 'bilinear (scipy.ndimage.affine_transform, order=1)', 'padding_hu': AR_HU,
    'windows_level_width_hu': JANELAS, 'image_format': 'PNG, 8-bit, 3 channels (brain, subdural, bone)',
    'training_classes': ['hemorrhage'],
    'cq500': resumo_base(img_cq, cx_cq, info_cq), 'cq500_folds_main_analysis': por_parte,
    'cq500_equivocal_slices': sum(i['grupo'] == 'duvidoso' for i in img_cq),
    'ct_ich': resumo_base(img_ct, cx_ct, info_ct), 'tests': testes,
}
print(json.dumps(resumo, indent=2))
(PREPARO / 'export_512_summary.json').write_text(json.dumps(resumo, indent=2))
with open(PREPARO / 'cq500_export_exams.csv', 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=info_cq[0].keys()); w.writeheader(); w.writerows(info_cq)

inicio = time.time()
for base in ('cq500', 'ctich'):
    destino = PREPARO / f'{base}_512.zip'
    temporario = LOCAL / f'{base}_512.zip'
    with zipfile.ZipFile(temporario, 'w', zipfile.ZIP_STORED) as z:        # PNG is already compressed
        for p in sorted((LOCAL / base).rglob('*')):
            if p.is_file():
                z.write(p, p.relative_to(LOCAL / base))
    shutil.copy(temporario, destino)
    print(destino.name, f'{destino.stat().st_size / 1e9:.2f} GB')
print(f'cópia para o Drive: {time.time() - inicio:.0f} s')

In [ ]:
# 8. Figure for the article: a CQ500 slice before and after preparation, with the boxes
import matplotlib
if 'google.colab' not in sys.modules:
    matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

EXEMPLO = 'CQ500-CT-173' if 'CQ500-CT-173' in nomes else nomes[0]
e = escolhidas[EXEMPLO]
with zipfile.ZipFile(zip_do_exame(EXEMPLO)) as z:
    dcms = sorted((pydicom.dcmread(io.BytesIO(z.read(m))) for m in z.namelist()
                   if m.lower().endswith('.dcm') and m.rsplit('/', 2)[-2] == e['pasta']),
                  key=lambda d: float(d.ImagePositionPatient[2]))
vol = np.stack([d.pixel_array * float(d.RescaleSlope) + float(d.RescaleIntercept) for d in dcms]).astype(np.float32)
info = next(i for i in info_cq if i['exame'] == EXEMPLO)
k = max(range(len(vol)), key=lambda i: len(caixas_bhx.get((EXEMPLO, i), [])))
centro, _ = centro_da_cabeca(vol)
escala, desloc = geometria(info['pixel_mm_original'], centro)
depois = np.asarray(Image.open(LOCAL / 'cq500' / 'images' / f'{EXEMPLO}_s{k:03d}.png'))

fig, eixos = plt.subplots(1, 2, figsize=(10, 5.2))
eixos[0].imshow(janelas(vol[k])[..., 0], cmap='gray')
for c in caixas_bhx.get((EXEMPLO, k), []):
    eixos[0].add_patch(Rectangle((float(c['x']), float(c['y'])), float(c['largura']), float(c['altura']),
                                 fill=False, color='yellow', lw=1))
lado = LADO / escala                                   # 25 cm crop drawn on the original image
eixos[0].add_patch(Rectangle(((desloc[1]) / escala, (desloc[0]) / escala), lado, lado, fill=False,
                             color='cyan', ls='--', lw=1.2))
eixos[0].set_title(f'Original: {info["pixel_mm_original"]:.3f} mm/pixel, brain window\n'
                   'yellow: BHX boxes; cyan: 250 mm crop centred on the head', fontsize=9)
eixos[1].imshow(depois)
for c in cx_cq:
    if c['exame'] == EXEMPLO and c['corte'] == k:
        eixos[1].add_patch(Rectangle((c['x1'], c['y1']), c['x2'] - c['x1'], c['y2'] - c['y1'], fill=False,
                                     color='yellow', lw=1))
eixos[1].set_title(f'Model input: {PIXEL_MM} mm/pixel, {LADO}×{LADO}\n'
                   'RGB = brain (40/80), subdural (80/200), bone (600/2800) windows', fontsize=9)
for ax in eixos:
    ax.axis('off')
fig.suptitle(f'{EXEMPLO}, slice {k + 1} of {len(vol)} · CQ500 (CC BY-NC-SA 4.0) with BHX boxes', fontsize=9)
fig.tight_layout()
for ext in ('png', 'pdf'):
    fig.savefig(PREPARO / f'f3-04-preprocessing.{ext}', dpi=150)
plt.show()

## After running

1. Cell 6 must finish without error (it stops the notebook if any test fails).
2. `tc/preparo/export_512_summary.json` and `tc/preparo/cq500_export_exams.csv` summarise the export; the
   archives stay on the Drive.